In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [3]:
bangladesh = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))

In [4]:
m = geemap.Map()
m.centerObject(bangladesh, 7)
m.set_options('HYBRID')

dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')

def simulate_rise(meters):
    #find pixels lower than the rise level
    flooded = dem.lt(meters)
    
    #remove the existing permanent water as we want only the new floodings
    #if occurrence > 50 means it is usually water 
    permanent_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence').gt(50)
    
    #now new mask gonna keep the flooded pixels where it is no permanent water
    new_flood = flooded.updateMask(permanent_water.Not())
    
    return new_flood.selfMask()



In [8]:
rise_2m = simulate_rise(2)
m2m = geemap.Map()
m2m.centerObject(bangladesh, 7)
m2m.set_options('HYBRID')
m2m.addLayer(rise_2m, {'palette': '0000FF'}, '2m')
m2m.add_legend(title="Sea Level Rise for 2m", keys=["Flooded Area"], colors=["0000FF"])
m2m


Map(center=[23.82705199221704, 90.28837386642488], controls=(WidgetControl(options=['position', 'transparent_b…

In [9]:
def rise_map(meters):
    rise = simulate_rise(meters)
    map_rise = geemap.Map()
    map_rise.centerObject(bangladesh, 7)
    map_rise.set_options('HYBRID')
    map_rise.addLayer(rise, {'palette': '0000FF'}, f'{meters}m')
    map_rise.add_legend(title=f"Sea Level Rise for {meters}m", keys=["Flooded Area"], colors=["0000FF"])
    return map_rise


#testing
rise_map(5)

Map(center=[23.82705199221704, 90.28837386642488], controls=(WidgetControl(options=['position', 'transparent_b…

In [11]:
rise_map(2)

Map(center=[23.82705199221704, 90.28837386642488], controls=(WidgetControl(options=['position', 'transparent_b…